## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from datetime import datetime
from astropy.time import Time
pd.set_option('mode.copy_on_write', True)
pd.set_option('future.no_silent_downcasting', True)

load_dotenv()

RAW_DATA_PATH = os.getenv('MAG_RAW_PATH')
TREATED_GLOBAL_PATH = os.getenv('MAG_TREATED_GLOBAL_PATH')
TREATED_BY_REGION_PATH = os.getenv('MAG_TREATED_BY_REGION_PATH')

BEGIN_DATE = "20100501_000000"
END_DATE = "20240921_235900"

SHARP_PARAMS = [
    'USFLUX', 'R_VALUE', 'TOTUSJH', 'TOTUSJZ',
    'ABSNJZH', 'SHRGT45', 'MEANPOT', 'TOTPOT',
    'MEANALP'
]

## Reading Data

In [ ]:
def parse_tai_to_utc(tai_series: pd.Series) -> pd.Series:
    iso_strings = tai_series.str.replace('.', '-', regex=False).str.replace('_', 'T', regex=False)

    t_tai = Time(iso_strings.tolist(), format='isot', scale='tai')
    return pd.to_datetime(t_tai.utc.isot)

In [ ]:
date_format = "%Y%m%d_%H%M%S"
start_dt = datetime.strptime(BEGIN_DATE, date_format)
end_dt = datetime.strptime(END_DATE, date_format)

df_raw_list = []
current_start = start_dt

while current_start < end_dt:
    current_end = current_start + pd.DateOffset(months=1)
    if current_end > end_dt:
        current_end = end_dt

    start_year = str(current_start.year)
    year_dir = os.path.join(RAW_DATA_PATH, start_year)

    start_str = current_start.strftime("%Y%m%d_%H%M%S")
    end_str = current_end.strftime("%Y%m%d_%H%M%S")

    file_name = f"jsoc_data_{start_str}_TAI_to_{end_str}_TAI.csv"
    full_path = os.path.join(year_dir, file_name)

    if os.path.exists(full_path):
        df = pd.read_csv(full_path)

        df.loc[:, 'REGION_ID'] = df['DATASET_QUERY'].str.extract(r'\[(\d+)\]')

        df.loc[:, 'T_REC'] = df['T_REC'].str.replace('_TAI', '', regex=False)
        df.loc[:, 'T_REC'] = parse_tai_to_utc(df['T_REC'])
        df = df.rename(columns={'T_REC': 'ds'})

        df = df.drop(columns=['EPSX'])

        df_raw_list.append(df)

    current_start = current_end

df_mag_raw = pd.concat(df_raw_list, ignore_index=True)

In [ ]:
df_mag_raw

## Treating Data

In [ ]:
df_clean = df_mag_raw.copy()

# 1. Filtro de Qualidade: Descartar onde QUALITY != 0
df_clean.loc[:, 'QUALITY'] = df_clean['QUALITY'].apply(
    lambda x: int(str(x), 16) if isinstance(x, str) and str(x).startswith('0x') else pd.to_numeric(x, errors='coerce')
)
df_clean = df_clean[df_clean['QUALITY'] == 0].copy()

cols_to_numeric = SHARP_PARAMS + ['LON_MIN', 'LON_MAX']
for col in cols_to_numeric:
    df_clean.loc[:, col] = pd.to_numeric(df_clean[col], errors='coerce')

# 3. Remover Valores Infinitos
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).infer_objects(copy=False)

# 4. Remover Dados Ausentes e Incompletos
df_clean = df_clean.dropna(subset=cols_to_numeric)

# 5. Filtro de Longitude: Bordas estritamente dentro de ±70°
df_clean = df_clean[(df_clean['LON_MIN'] >= -70) & (df_clean['LON_MAX'] <= 70)].copy()

# 6. Remover registros sem ID de Região e consolidar a cópia final
df_clean = df_clean.dropna(subset=['REGION_ID']).copy()

ds_new = df_clean['ds'].dt.tz_localize('UTC').dt.round('12min')
df_clean = df_clean.drop(columns=['ds']).copy()
df_clean.loc[:, 'ds'] = ds_new

In [ ]:
df_clean.head().to_csv()